# GNR653 Lecture 1: Reproducible research and the shell

**Tue 22 Sep 2026, 14:00-15:25.** Keep this notebook open on the left and a terminal on the right (drag the terminal panel to the side of the editor).

**Before you start** (in the terminal, from the repo root):
```bash
bash scripts/update_from_template.sh
git push
```
Everything for this lecture lives in `lectures/L01/`: this notebook, the data, your scratch work and your in-class assignment. From section 3 on, all terminal commands are run from inside `lectures/L01/`.

## By the end of today you can
- explain why a scripted workflow is more reproducible than point-and-click GIS;
- find your way around the codespace: editor, terminal, file explorer;
- use absolute and relative paths, and move around with `pwd`, `ls`, `cd`;
- manage files with `mkdir`, `cp`, `mv`, `rm` and wildcards;
- look inside text files with `cat`, `head`, `tail`, `wc`, `grep`, and chain commands with pipes and redirection;
- explain why pipes work (small tools, text streams, stdin/stdout/stderr) and use exit codes with `$?`, `&&` and `||`;
- write and run a small bash script with a `for` loop.

## Plan
| Time | What |
|---|---|
| 0-5 | Resume codespace, update, run cell 1 |
| 5-53 | Live coding: sections 2 to 7 |
| 53-80 | In-class assignment IC1 (section 8) |
| 80-85 | Commit, push, submit the commit link on Moodle |

**How to use this notebook.** Most of today happens in the **terminal**. Commands in grey boxes are for you to type there (typing them yourself is the point; use Tab to complete names). Cells marked *check* run the answer for you to compare. Answers to "your turn" questions are hidden under **Show answer**: try first.

## 1. Get today's data

Run the cell below (Shift+Enter). It
1. checks the original files in `lectures/L01/original/` against their checksums (`SHA256SUMS`),
2. copies them to `lectures/L01/work/scenes/` and `lectures/L01/work/messy/`, where you will work,
3. makes sure `work/` and `sandbox/` (our scratch folder) are ignored by git.

Nothing you do in `work/` or `sandbox/` gets committed. **Broke something? Run this cell again** and you get a fresh copy. Never edit `original/`.

In [ ]:
# Cell 1: verify and copy today's data. Safe to rerun at any time.
import hashlib
import shutil
import subprocess
from pathlib import Path

repo = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"],
                           capture_output=True, text=True, check=True).stdout.strip())
lec = repo / "lectures" / "L01"
src = lec / "original"
if not (src / "SHA256SUMS").exists():
    raise FileNotFoundError("lectures/L01/original is missing. In the terminal, from the repo root, run:\n"
                            "    bash scripts/update_from_template.sh")

# 1. verify the originals
lines = (src / "SHA256SUMS").read_text().splitlines()
bad = []
for line in lines:
    digest, name = line.split("  ", 1)
    f = src / name
    if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != digest:
        bad.append(name)
if bad:
    print("WARNING: these original files are missing or changed:", *bad, sep="\n  ")
    print("Restore them with: git checkout -- lectures/L01/original")
else:
    print(f"{len(lines)} original files verified against SHA256SUMS")

# 2. fresh working copies in work/
for sub in ["scenes", "messy"]:
    target = lec / "work" / sub
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(src / sub, target)
    print(f"{len(list(target.iterdir())):4d} files copied to lectures/L01/work/{sub}/")

# 3. keep working copies and scratch files out of git
gi = repo / ".gitignore"
text = gi.read_text() if gi.exists() else ""
missing = [p for p in ["work/", "sandbox/"] if p not in text.split()]
if missing:
    gi.write_text(text.rstrip("\n") + "\n\n# Your working copies and scratch folders, e.g. lectures/L01/work/ (never committed)\n" + "\n".join(missing) + "\n")
    print("added to .gitignore:", " ".join(missing))

print("\nRepo root:", repo)

### Where did these files come from?
Real metadata for **every** Sentinel-2 L2A and Landsat 8/9 Collection 2 Level-2 scene over Mumbai in 2025: 88 Sentinel-2 scenes (tile T43QBB) and 46 Landsat scenes (path/row 148/047). A script queried the Earth Search STAC catalogue for them; nobody clicked through a download portal. You will query STAC yourself in Lecture 6.

- The **file names** are the real product names.
- The **contents** are a small, simplified subset of the real metadata, reformatted so we can practise on them. Sentinel-2 files use `KEY=VALUE` lines; Landsat files imitate the MTL layout.
- A few `.xml` files and the `messy/` folder are there on purpose, to make the exercises realistic.
- Licences: contains modified Copernicus Sentinel data 2025; Landsat data courtesy of the U.S. Geological Survey. See `lectures/L01/original/README.md`.

The same checksum test from the shell (optional): `cd lectures/L01/original && sha256sum -c --quiet SHA256SUMS && cd -`

## 2. Where am I? Paths and navigation

The file system is a tree. In your codespace:
```
/
|-- home/
|   `-- vscode/                 ~  (your home)
`-- workspaces/
    `-- gnr653-2026-<roll>/     repo root
        |-- .venv/              hidden (starts with a dot)
        |-- lectures/
        |   `-- L01/            everything for Lecture 1
        |       |-- L01_shell.ipynb
        |       |-- original/   the data as shipped (do not edit)
        |       |-- work/       your working copy (ignored by git)
        |       |   |-- scenes/
        |       |   `-- messy/
        |       |-- sandbox/    scratch (ignored by git)
        |       `-- ic01/       your in-class assignment
        `-- scripts/
```
- An **absolute** path starts at `/`: `/workspaces/gnr653-2026-<roll>/lectures/L01/work`
- A **relative** path starts where you are: from `lectures/L01`, that folder is just `work`
- `.` is here, `..` is the parent folder, `~` is your home folder.

**In the terminal**, start at the repo root and type these one at a time, reading the output:
```bash
pwd                          # print working directory: where am I?
ls                           # list what is here
ls -la                       # long listing, including hidden files
cd lectures                  # go into a folder
pwd
cd ..                        # up one level
cd ~                         # home
cd -                         # back to where you were
cd /workspaces/*             # back to the repo, from anywhere
cd lectures/L01              # where we work from now on
ls -lh work/scenes | head
```
Help: `ls --help`, or `man ls` (press `q` to quit). Use **Tab** to complete names and the **up arrow** for previous commands.

### Why not run shell commands in the notebook?
A notebook line starting with `!` runs in a **new shell that exits immediately**. Run the next cell and look at the two `pwd` results.

In [ ]:
!pwd
!cd ..
!pwd

The `cd ..` happened in a shell that closed straight away, so the second `pwd` is unchanged. IPython has a special command, `%cd`, that changes the notebook's own folder:

In [ ]:
%cd ..
%cd L01

This is why we do shell work in the **terminal**: it keeps its state between commands.

### Your turn 2.1
Go to `lectures/` (one level above `L01`). From there, list the files in `L01/work/scenes`
1. with a relative path,
2. with an absolute path.

Then list `L01/original/scenes`. Both folders hold the same files: which one should you experiment on? Finish with `cd L01`.

<details><summary>Show answer</summary>

```bash
cd ..                                                       # from lectures/L01 to lectures/
ls L01/work/scenes
ls /workspaces/gnr653-2026-<roll>/lectures/L01/work/scenes  # use Tab to fill in your repo name
ls L01/original/scenes
cd L01
```
Experiment on `work/scenes`: it is a copy that cell 1 can recreate. `original/` is what the course shipped.
</details>

## 3. Creating, copying, moving, deleting

| Command | What it does |
|---|---|
| `mkdir -p a/b` | make folder `a/b` (and `a` if needed; no error if it exists) |
| `cp src dest` | copy a file |
| `cp -r src dest` | copy a folder and everything in it |
| `mv old new` | move **or** rename (the same thing to the shell) |
| `rm file` | delete a file |
| `rm -r folder` | delete a folder and everything in it |

**There is no recycle bin.** `rm` is permanent. Habit: run `ls` with exactly the same pattern first, check the list, then change `ls` to `rm`.

**In the terminal**, from `lectures/L01` (`sandbox/` is our scratch folder; git ignores it):
```bash
mkdir -p sandbox/raw
cp work/scenes/S2A_*_202501*.txt sandbox/raw/
ls sandbox/raw
cp -r sandbox/raw sandbox/raw_backup
mv sandbox/raw/S2A_MSIL2A_20250101T054231_N0511_R005_T43QBB_20250101T085751.txt sandbox/raw/first_scene.txt
ls sandbox/raw
rm sandbox/raw/first_scene.txt
rm -r sandbox/raw_backup
```
(Type the first few letters of a long name and press Tab.)

### Your turn 3.1
Copy all Landsat 9 metadata files into a new folder `sandbox/landsat/` with **one** `cp` command. How many files did you copy?

<details><summary>Show answer</summary>

```bash
mkdir -p sandbox/landsat
cp work/scenes/LC09_*_MTL.txt sandbox/landsat/
ls sandbox/landsat | wc -l      # 23
```
</details>

## 4. Wildcards and awkward file names

| Pattern | Matches | Example |
|---|---|---|
| `*` | any characters, including none | `ls work/scenes/S2*.txt` |
| `?` | exactly one character | `ls work/scenes/S2?_MSIL2A_202507*` |
| `[AB]` | one character from the set | `ls work/scenes/S2[AB]_*.txt` |

The **shell** expands the pattern **before** the command runs. `echo` shows you what the command will receive:
```bash
echo work/scenes/S2*
ls work/scenes/S2* | wc -l
ls work/scenes/S2*.txt | wc -l
```
Why do the two counts differ? (Look for files that are not `.txt`.)

### Names with spaces
A space separates arguments, so a file name with spaces looks like several names:
```bash
ls work/messy
cat work/messy/S2B scene copy.txt       # fails: cat receives 3 names
cat "work/messy/S2B scene copy.txt"     # works: quotes keep it together
ls work/messy/*.txt                     # which file is missing, and why?
```
Patterns are case sensitive: `*.txt` does not match `.TXT`. Lesson: never put spaces in your own file names.

### Your turn 4.1
With **one** `ls` command, list only the Sentinel-2 `.txt` files from July and August 2025. How many are there?

<details><summary>Show answer</summary>

```bash
ls work/scenes/S2?_MSIL2A_20250[78]*.txt
ls work/scenes/S2?_MSIL2A_20250[78]*.txt | wc -l     # 15 (July 7, August 8)
```
</details>

### Predict the output
In `work/scenes`, how many names does `ls S2?_MSIL2A_202507*` print? Decide, then run it.

<details><summary>Show answer</summary>

8: seven July scenes plus one `_MTD_TL.xml` file, because the pattern does not end in `.txt`.
</details>

## 5. Looking inside text files

| Command | What it does |
|---|---|
| `cat file` | print the whole file |
| `head -n 3 file` / `tail -n 2 file` | first 3 / last 2 lines |
| `wc -l file` | count lines |
| `grep PATTERN files` | print the lines that contain PATTERN |
| `grep -c` / `-l` / `-v` / `-i` | count matches / list matching files / invert / ignore case |

**In the terminal**, from `lectures/L01`:
```bash
cd work/scenes
cat S2A_MSIL2A_20250101T054231_N0511_R005_T43QBB_20250101T085751.txt
cat LC09_L2SP_148047_20250112_02_T1_MTL.txt
head -n 3 LC09_L2SP_148047_20250112_02_T1_MTL.txt       # not very useful here: why?
wc -l S2A_MSIL2A_20250101T054231_N0511_R005_T43QBB_20250101T085751.txt LC09_L2SP_148047_20250112_02_T1_MTL.txt
grep CLOUD_COVER S2A_MSIL2A_20250101T054231_N0511_R005_T43QBB_20250101T085751.txt
grep CLOUD_COVER LC09_L2SP_148047_20250112_02_T1_MTL.txt  # two lines: why?
cd ../..
```
Same information, two formats: `CLOUD_COVER=0.01` (Sentinel-2, our simplified files) and `CLOUD_COVER = 0.33` (Landsat MTL layout, with spaces). A pattern must match the real format: `grep "CLOUD_COVER="` finds nothing in the Landsat files.

### Your turn 5.1
How many Sentinel-2 scenes come from the satellite Sentinel-2B? Answer it twice: once from the file **contents** (`grep`), once from the file **names** (`ls`). Do the answers agree?

<details><summary>Show answer</summary>

```bash
grep -l "Sentinel-2B" work/scenes/S2*.txt | wc -l     # 36
ls work/scenes/S2B* | wc -l                            # 37
ls work/scenes/S2B*.txt | wc -l                        # 36
```
`S2B*` also matches one `.xml` file. When two methods disagree, look closer: that is how you find bugs.
</details>

### Your turn 5.2
Landsat scenes come in Tier 1 (best quality) and Tier 2. How many of our Landsat scenes are Tier 2, and in which months?

<details><summary>Show answer</summary>

```bash
grep -l '"T2"' work/scenes/L*_MTL.txt | wc -l       # 11
ls work/scenes/*_T2_MTL.txt                          # from the names: all between May and September
```
</details>

## 6. Pipes and redirection

Every command reads **standard input** and writes **standard output**; errors go to **standard error**.

| Operator | Meaning |
|---|---|
| `a \| b` | send the output of `a` into `b` |
| `a > file` | write the output to a file (**overwrites** it) |
| `a >> file` | append the output to a file |
| `a 2> errors.txt` | write error messages to a file |

Small tools that each do one job, joined by pipes: that is the Unix idea.

### Which 5 Sentinel-2 scenes are the clearest?
Build the pipeline one stage at a time, in the terminal, and run it after each stage:
```bash
cd work/scenes
grep CLOUD_COVER S2*.txt
grep CLOUD_COVER S2*.txt | sort -t= -k2 -n
grep CLOUD_COVER S2*.txt | sort -t= -k2 -n | head -n 5
grep CLOUD_COVER S2*.txt | sort -t= -k2 -n | head -n 5 > ../../sandbox/clearest.txt
cat ../../sandbox/clearest.txt
```
`sort -t= -k2 -n` means: split each line at `=`, sort on the 2nd field, as numbers. All five have `CLOUD_COVER=0.00`. What does "the clearest" mean when there is a tie? (`sort` then compares the whole lines.)

### How many scenes per month?
The sensing time is field 3 of the name when you split at `_`; its first 6 characters are `YYYYMM`.
```bash
ls S2*.txt | cut -d_ -f3 | cut -c1-6 | sort | uniq -c
```

### How many cloud-free scenes per month?
`CLOUD_COVER=0.` matches only values below 1 (the `\` makes `.` a literal dot):
```bash
grep -l "CLOUD_COVER=0\." S2*.txt | cut -d_ -f3 | cut -c1-6 | sort | uniq -c
cd ../..
```
Compare the two lists. The satellites image Mumbai every few days, all year round. So why are May to September missing from the second list? What does that mean if you want to map floods with optical images during the monsoon?

### Your turn 6.1
Run the sort without `-n`, in reverse order, and explain why `9x.xx` values come before `100.00`:
```bash
grep CLOUD_COVER work/scenes/S2*.txt | sort -t= -k2 -r | head
```
<details><summary>Show answer</summary>

Without `-n`, `sort` compares text character by character. `"9"` sorts after `"1"`, so `99.98` is "bigger" than `100.00`. Add `-n` to compare numbers.
</details>

*Check*: run this cell to compare with your terminal output.

In [ ]:
%%bash
cd "$(git rev-parse --show-toplevel)/lectures/L01/work/scenes"
echo "== 5 clearest (all tied at 0.00)"
grep CLOUD_COVER S2*.txt | sort -t= -k2 -n | head -n 5
echo
echo "== Sentinel-2 scenes per month"
ls S2*.txt | cut -d_ -f3 | cut -c1-6 | sort | uniq -c
echo
echo "== cloud-free (below 1%) scenes per month"
grep -l "CLOUD_COVER=0\." S2*.txt | cut -d_ -f3 | cut -c1-6 | sort | uniq -c

### Why do pipes work? The Unix philosophy
Doug McIlroy proposed the pipe at Bell Labs, and it arrived in Unix in 1973. The pipelines above work because Unix tools follow a few design rules:
- **Do one thing well.** `grep` filters lines, `sort` orders them, `uniq` collapses repeats, `wc` counts. None of them tries to do everything; complex tasks come from combining small tools.
- **Text is the common interface.** Every tool reads lines of text and writes lines of text, so the output of any tool can become the input of any other, without either knowing about the other.
- **Three standard streams.** Every program starts with standard input (stdin), standard output (stdout) and standard error (stderr) already open. The shell decides where they point: the terminal, a file (`<`, `>`), or another program (`|`). The program does not care. Errors go to stderr, so they do not get mixed into the data flowing down a pipe.
- **Everything is a file.** Files, devices and pipes are all read and written in the same way. A pipe is a small buffer inside the operating system that looks like a file to both ends, so a program needs no special code to be part of a pipeline.

### Did it work? Exit codes
Every command also returns an **exit code** when it finishes: `0` means success, anything else means failure. The shell keeps the last one in `$?`. Try this in the terminal, from `lectures/L01`:
```bash
ls work/scenes > /dev/null     # /dev/null throws the output away
echo $?                            # 0: it worked
ls no_such_folder
echo $?                            # 2: it failed; the message went to stderr
ls no_such_folder > out.txt        # the error still appears on screen
ls no_such_folder 2> err.txt       # now it goes into err.txt
cat err.txt
rm out.txt err.txt
```
The shell can use exit codes to decide what to run next:

| Operator | Meaning |
|---|---|
| `a && b` | run `b` only if `a` succeeded |
| `a \|\| b` | run `b` only if `a` failed |

```bash
grep -q "CLOUD_COVER=100" work/scenes/S2*.txt && echo "some scenes are fully cloudy"
grep -q "CLOUD_COVER=-" work/scenes/S2*.txt || echo "no negative cloud cover: good"
```
`grep -q` prints nothing; it only sets the exit code. The second line is a one-line sanity check on the data. In section 7, `set -euo pipefail` uses the same idea to stop a script at the first command that fails.

### Your turn 6.2
1. After `ls no_such_folder > out.txt`, what is inside `out.txt`, and why?
2. Predict what this prints, then run it. Then change `*.xml` to `*.csv` and predict again.
```bash
ls work/scenes/*.xml > /dev/null && echo "found" || echo "none found"
```
<details><summary>Show answer</summary>

1. Nothing. The shell creates (empties) `out.txt` before `ls` runs, and the error message goes to stderr, which `>` does not redirect. `2>` redirects stderr.
2. `found`: there are 4 `.xml` files, so `ls` succeeds and the `&&` part runs. With `*.csv` there is no match, `ls` prints an error and fails, so the `&&` part is skipped and the `||` part prints `none found`.
</details>

*Check*: run this cell to compare with your terminal output.

In [ ]:
%%bash
cd "$(git rev-parse --show-toplevel)/lectures/L01"
ls work/scenes > /dev/null
echo "exit code of a working ls: $?"
ls no_such_folder 2> /dev/null
echo "exit code of a failing ls: $?"
grep -q "CLOUD_COVER=100" work/scenes/S2*.txt && echo "some scenes are fully cloudy"
grep -q "CLOUD_COVER=-" work/scenes/S2*.txt || echo "no negative cloud cover: good"
ls work/scenes/*.xml > /dev/null && echo "*.xml: found" || echo "*.xml: none found"
ls work/scenes/*.csv > /dev/null 2>&1 && echo "*.csv: found" || echo "*.csv: none found"

## 7. Scripts and loops

A script is a text file of commands. Create `sandbox/clearest.sh` in the editor (File > New File) with:
```bash
#!/usr/bin/env bash
# clearest.sh: the 5 clearest Sentinel-2 scenes
set -euo pipefail          # stop at the first error, unset variable, or failed pipe stage

cd work/scenes
grep CLOUD_COVER S2*.txt \
    | sort -t= -k2 -n \
    | head -n 5
```
Run it from `lectures/L01` with `bash sandbox/clearest.sh`. Your typed commands have become a **record**: you can rerun it, review it and commit it.

### Variables and `for` loops
```bash
name="S2A_MSIL2A_20250101T054231_N0511_R005_T43QBB_20250101T085751.txt"   # no spaces around =
echo "$name"                                   # use it with $, inside double quotes
sat=$(echo "$name" | cut -d_ -f1)              # $(...) runs a command and captures its output
echo "$sat"

for f in work/scenes/S2A*.txt; do          # f takes each file name in turn
    echo "$f"
done
```

### Worked example: rename (copy) every scene to a date-first name
A Sentinel-2 name has 7 fields: `S2A` `MSIL2A` `20250101T054231` `N0511` `R005` `T43QBB` `20250101T085751.txt` (satellite, product, sensing time, baseline, orbit, tile, processing time). We want `20250101_S2A_T43QBB.txt`.

Save this as `sandbox/rename_demo.sh`:
```bash
#!/usr/bin/env bash
set -euo pipefail
mkdir -p sandbox/renamed
for f in work/scenes/S2*.txt; do
    name=$(basename "$f")                                # drop the folder part
    sat=$(echo "$name" | cut -d_ -f1)
    date=$(echo "$name" | cut -d_ -f3 | cut -c1-8)
    tile=$(echo "$name" | cut -d_ -f6)
    new="${date}_${sat}_${tile}.txt"
    echo cp "$f" "sandbox/renamed/$new"                  # dry run: print, do not copy
done
```
Run `bash sandbox/rename_demo.sh | head`. When the printed commands look right, delete the word `echo` and run it again. Always dry-run a loop before it copies, moves or deletes anything.

### Your turn 7.1
Change the loop so that it also prints each scene's cloud cover next to its new name, e.g. `20250101_S2A_T43QBB.txt 0.01`.

<details><summary>Show answer</summary>

Add inside the loop:
```bash
    cloud=$(grep CLOUD_COVER "$f" | cut -d= -f2)
    echo "$new $cloud"
```
</details>

**Stretch:** do the same for the Landsat files. Their date is field 4 and the satellite is field 1: `20250112_LC09_148047.txt`.

## 8. In-class assignment IC1: scene metadata report (53-80, 10 points)

Individual work. Notes and the web are allowed; copying from classmates is not. Work in `lectures/L01`, on the files in `work/scenes/` (rerun cell 1 if you damaged them). Put everything in a new folder `lectures/L01/ic01/`.

**Task A (4 points).** Write `ic01/scene_report.sh`. When run from `lectures/L01` as `bash ic01/scene_report.sh`, it must write `ic01/report.txt` containing, each part preceded by a heading line:
1. the number of Sentinel-2 metadata files and the number of Landsat metadata files (`.txt` files only, not `.xml`);
2. the number of Sentinel-2 scenes per month;
3. the number of cloud-free (cloud cover below 1%) Sentinel-2 scenes per month.

**Task B (3 points).** In the same script, add a `for` loop that copies every Sentinel-2 `.txt` file into `ic01/renamed/` as `YYYYMMDD_<sat>_<tile>.txt`, e.g. `20250101_S2A_T43QBB.txt`. Running the script twice must not fail.

**Task C (2 points).** Create `ic01/answers.md` and answer in 2-3 sentences each:
1. Why is your script more reproducible than doing the same steps by hand in a file browser?
2. Which paths in your script are relative, and relative to what? What breaks if someone runs it from inside `ic01/`?
3. Every month has 5 to 8 Sentinel-2 scenes, but cloud-free scenes vanish from May to September. Why, and what does this imply for monitoring floods with optical satellites?

**Task D (1 point).** Commit, push, and paste the commit link on Moodle (IC1) **by 15:25**. From `lectures/L01`:
```bash
git add ic01
git commit -m "IC1 scene report"
git push
```
To get the link: open your repo on GitHub, click the latest commit, copy the address from the browser (it ends in a long hash). We will see what these three git commands do in Lecture 2.

**Hints.** Build and test each pipeline in the terminal first, then paste it into the script. `echo "== heading" > ic01/report.txt` starts the file; `>>` adds to it. Check your output with `cat ic01/report.txt`.

**Stretch (no marks).** Make the script work from any folder (`cd "$(dirname "$0")/.."` at the top). List the scenes with cloud cover below 20% using `awk -F= '$2 < 20'`. Add Landsat scenes per month.

## 9. Summary

| Command | What it does | Example from today |
|---|---|---|
| `pwd`, `ls`, `cd` | where am I, what is here, go somewhere | `ls -lh work/scenes \| head` |
| `mkdir -p`, `cp`, `mv`, `rm` | make, copy, move or rename, delete | `cp work/scenes/LC09_*_MTL.txt sandbox/landsat/` |
| `*`, `?`, `[AB]` | wildcards, expanded by the shell | `ls work/scenes/S2?_MSIL2A_20250[78]*.txt` |
| `cat`, `head`, `tail`, `wc` | look at and count text | `wc -l work/scenes/*.txt` |
| `grep` | find lines that match | `grep -l "Sentinel-2B" work/scenes/S2*.txt` |
| `sort`, `uniq -c`, `cut` | sort, count repeats, cut out fields | `ls S2*.txt \| cut -d_ -f3 \| cut -c1-6 \| sort \| uniq -c` |
| `\|`, `>`, `>>`, `2>` | pipe, write, append, write errors | `... \| head -n 5 > sandbox/clearest.txt` |
| `$?`, `&&`, `\|\|` | exit code of the last command; run next only on success / on failure | `grep -q "CLOUD_COVER=-" work/scenes/S2*.txt \|\| echo "none"` |
| `for ... do ... done` | repeat for each file | `for f in work/scenes/S2*.txt; do echo "$f"; done` |

**Next lecture (Fri 25 Sep): git and GitHub.** What `add`, `commit` and `push` actually did to your IC1.

**Reading:** Software Carpentry, [The Unix Shell](https://swcarpentry.github.io/shell-novice/), episodes 4 to 6, and [Version Control with Git](https://swcarpentry.github.io/git-novice/), episodes 1 to 4.